In [1]:
!uv pip install --extra-index-url https://registry.partner.nextopia.dev/index/ -q "nxd.data_product[notebook]"

In [ ]:
# Pre-req: pandas and requests
# !pip install pandas requests

import io

import nxd.data_product.context as ctx
import pandas as pd
import requests
from nxd.data_product.client import create_client

nxd_client = create_client(hostname="dp.partner.nextopia.dev")
data_product = nxd_client.data_product(data_product="payments-demo")
output = data_product.get_access_credentials("object-storage", ctx.S3Input)


def get_dataframe(output: ctx.S3Input, model_name: str) -> pd.DataFrame:
    signed_url = output.model_urls[model_name]
    response = requests.get(signed_url)
    check_response_status(output, model_name, response)
    stream_io = io.BytesIO(response.content)
    return pd.read_csv(stream_io, encoding="utf-8")


def check_response_status(output: ctx.S3Input, model_name: str, response: requests.models.Response):
    if response.status_code != 200:
        s3_key = output.model_output_paths[model_name]
        error_message = f"Failed to fetch {s3_key} from the signed URL. Status code: {response.status_code}. Reason: {response.reason}"
        raise Exception(error_message)


data = get_dataframe(output, "payments_model")
data.head()

,customer,age,zipcode,merchant,zipmerchant,category,amount,fraud
0,'C1093826151',NaN,NaN,'M348934600',NaN,'es_transportation',4.55,False
1,'C352968107',NaN,NaN,'M348934600',NaN,'es_transportation',39.68,False
2,'C2054744914',NaN,NaN,'M1823072687',NaN,'es_transportation',26.89,False
3,'C1760612790',NaN,NaN,'M348934600',NaN,'es_transportation',17.25,False
4,'C757503768',NaN,NaN,'M348934600',NaN,'es_transportation',35.72,False


### Read data from income-statements data product

In [ ]:
# Pre-req: for adls you need azure client sdk
# !pip install azure-storage-blob azure-identity

import io

import nxd.data_product.context as ctx
import pandas as pd
from azure.identity import ClientSecretCredential
from azure.storage.blob import BlobServiceClient
from nxd.data_product.client import create_client

nxd_client = create_client(hostname="dp.partner.nextopia.dev")
data_product = nxd_client.data_product(data_product="income-statements-demo")
adls_output = data_product.get_access_credentials("adls", ctx.AzureDataLakeStorage)


def get_dataframe(adls_output: ctx.AzureDataLakeStorage, model_name: str) -> pd.DataFrame:
    credential = ClientSecretCredential(adls_output.tenant_id, adls_output.client_id, adls_output.client_secret)
    blob_service_client = BlobServiceClient(
        account_url=f"https://{adls_output.account_name}.blob.core.windows.net", credential=credential
    )

    client = blob_service_client.get_container_client(container=adls_output.container)
    blob_key = adls_output.model_paths[model_name].path
    parquet_bytes = client.download_blob(blob_key).readall()
    return pd.read_parquet(io.BytesIO(parquet_bytes))


data = get_dataframe(adls_output, "income_statement_analytics")
print(data.head())

                                              metric       date         value  \
0                        tax_effect_of_unusual_items 2024-09-30 -3.454080e+07   
1                                 tax_rate_for_calcs 2024-09-30  3.084000e-01   
2                                total_unusual_items 2024-09-30 -1.120000e+08   
3             total_unusual_items_excluding_goodwill 2024-09-30 -1.120000e+08   
4  net_income_from_continuing_operation_net_minor... 2024-09-30  6.990000e+09   

   symbol  
0  WBC.AX  
1  WBC.AX  
2  WBC.AX  
3  WBC.AX  
4  WBC.AX  


In [4]:
# Pre-req: pandas and requests
# !pip install pandas requests

import io

import nxd.data_product.context as ctx
import pandas as pd
import requests
from nxd.data_product.client import create_client

nxd_client = create_client(hostname="dp.partner.nextopia.dev")
data_product = nxd_client.data_product(data_product="market-fraud-density-demo")
output = data_product.get_access_credentials("object-storage", ctx.S3Input)


def get_dataframe(output: ctx.S3Input, model_name: str) -> pd.DataFrame:
    signed_url = output.model_urls[model_name]
    response = requests.get(signed_url)
    check_response_status(output, model_name, response)
    stream_io = io.BytesIO(response.content)
    return pd.read_csv(stream_io, encoding="utf-8")


def check_response_status(output: ctx.S3Input, model_name: str, response: requests.models.Response):
    if response.status_code != 200:
        s3_key = output.model_output_paths[model_name]
        error_message = f"Failed to fetch {s3_key} from the signed URL. Status code: {response.status_code}. Reason: {response.reason}"
        raise Exception(error_message)


data = get_dataframe(output, "fraud_density_model")
print(data.head())

          market_type  transaction_count  fraud_event_count  fraud_rate_pct
0        'es_leisure'                499                474           94.99
1         'es_travel'                728                578           79.40
2  'es_sportsandtoys'               4002               1982           49.53
3  'es_hotelservices'               1744                548           31.42
4  'es_otherservices'                912                228           25.00


In [12]:
# Pre-req: pandas and requests
# !pip install pandas requests

import io

import nxd.data_product.context as ctx
import pandas as pd
import requests
from nxd.data_product.client import create_client

nxd_client = create_client(hostname="dp.partner.nextopia.dev")
data_product = nxd_client.data_product(data_product="income-statement-analytics-demo")
output = data_product.get_access_credentials("s3", ctx.S3Input)


def get_dataframe(output: ctx.S3Input, model_name: str) -> pd.DataFrame:
    signed_url = output.model_urls[model_name]
    response = requests.get(signed_url)
    check_response_status(output, model_name, response)
    stream_io = io.BytesIO(response.content)
    # return pd.read_csv(stream_io, encoding="utf-8")
    return pd.read_parquet(stream_io)


def check_response_status(output: ctx.S3Input, model_name: str, response: requests.models.Response):
    if response.status_code != 200:
        s3_key = output.model_output_paths[model_name]
        error_message = f"Failed to fetch {s3_key} from the signed URL. Status code: {response.status_code}. Reason: {response.reason}"
        raise Exception(error_message)


data = get_dataframe(output, "company_metadata")
print(data.head())

   symbol                 company_name              sector  \
0  WBC.AX  Westpac Banking Corporation  Financial Services   
1  ANZ.AX   ANZ Group Holdings Limited  Financial Services   
2  MQG.AX      Macquarie Group Limited  Financial Services   

              industry    market_cap currency  
0  Banks - Diversified  1.468197e+11      AUD  
1  Banks - Diversified  1.183953e+11      AUD  
2      Capital Markets  7.977558e+10      AUD  


In [11]:
data.head(-1)

,symbol,year,total_revenue,net_income,ebit,ebitda,gross_profit,operating_expense,net_profit_margin,yoy_revenue_growth,sector,industry,peer_median_revenue,revenue_peer_percentile
0,ANZ.AX,2022,1.900400e+10,7.119000e+09,NaN,NaN,NaN,9.478000e+09,0.374605,NaN,Financial Services,Banks - Diversified,1.900400e+10,0.666667
1,ANZ.AX,2023,2.018900e+10,7.106000e+09,NaN,NaN,NaN,9.866000e+09,0.351974,0.062355,Financial Services,Banks - Diversified,2.018900e+10,0.666667
2,ANZ.AX,2024,2.035500e+10,6.535000e+09,NaN,NaN,NaN,1.039800e+10,0.321051,0.008222,Financial Services,Banks - Diversified,2.035500e+10,0.666667
3,ANZ.AX,2025,2.230700e+10,5.891000e+09,NaN,NaN,NaN,1.181400e+10,0.264088,0.095898,Financial Services,Banks - Diversified,1.454850e+10,1.000000
4,MQG.AX,2022,6.887000e+09,4.706000e+09,8.004000e+09,8.584000e+09,6.887000e+09,1.090700e+10,0.683316,NaN,Financial Services,Capital Markets,1.900400e+10,0.333333
5,MQG.AX,2023,6.558000e+09,5.182000e+09,1.410300e+10,1.476100e+10,6.558000e+09,1.291800e+10,0.790180,-0.047771,Financial Services,Capital Markets,2.018900e+10,0.333333
6,MQG.AX,2024,6.249000e+09,3.522000e+09,1.785900e+10,1.858200e+10,6.249000e+09,1.239700e+10,0.563610,-0.047118,Financial Services,Capital Markets,2.035500e+10,0.333333
7,MQG.AX,2025,6.790000e+09,3.715000e+09,1.969700e+10,2.051500e+10,6.790000e+09,1.287700e+10,0.547128,0.086574,Financial Services,Capital Markets,1.454850e+10,0.500000
8,WBC.AX,2021,2.102000e+10,5.458000e+09,NaN,NaN,NaN,NaN,0.259657,NaN,Financial Services,Banks - Diversified,2.102000e+10,1.000000
9,WBC.AX,2022,2.042700e+10,5.694000e+09,NaN,NaN,NaN,NaN,0.278749,-0.028211,Financial Services,Banks - Diversified,1.900400e+10,1.000000
